<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/02_strings_sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Strings & biological sequences

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: Python* - Instructor: Anderson Santos

## Learning objectives
- format output with **f-strings** and `print` options;
- use core **string methods** (`upper`, `replace`, `count`, `find`, `split`, `join`);
- **index** and **slice** sequences, including the reverse-complement trick;
- understand that strings are **immutable**;
- **transcribe** DNA→RNA and read codons.

> **How to read this notebook while teaching.** Every code cell is commented: the lines starting with `#` explain what the code does and why it matters biologically, so you can narrate the class straight from the cell even without notes. Blockquotes marked **Instructor note** are extra talking points.

---
| Notebook | Topic |
|---|---|
| 01 | Python essentials & operators |
| 02 | Strings & biological sequences |
| 03 | Data structures |
| 04 | Control flow & functions |
| 05 | Files, scripting & modules |
| 06 | Pandas for omics |

## Setup — run this cell first

This cell makes the example data available whether you are on **your own computer** or on **Google Colab**. You do not need to understand it. Click it and press **Shift+Enter**.

> **Instructor note.** Set `GITHUB_REPO` below to your repository once, before sharing the notebooks.

In [1]:
# Run me first. Works on a local install AND on Google Colab.
import os

GITHUB_REPO = "andersonavilasantos/ufz-summerschool-python"   # <-- set to your GitHub repo

if not os.path.exists("../data/asv_table.csv"):
    # Data not found locally -> assume Google Colab and download the course files.
    os.system(f"git clone -q https://github.com/{GITHUB_REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")

assert os.path.exists("../data/asv_table.csv"), (
    "Could not find the data. On Colab, check GITHUB_REPO above; "
    "locally, run the notebook from inside the notebooks folder.")
print("✅ Setup complete — the data folder is available.")

✅ Setup complete — the data folder is available.


## 1. Printing & f-strings
`print` takes several values and options `sep` and `end`.

In [2]:
# print() accepts several values and two useful options:
print('A', 'C', 'G', 'T', sep=' | ')   # 'sep' sets what goes BETWEEN values
print('no newline here ->', end=' ')    # 'end' replaces the default line break
print('...continued on same line')      # so this prints on the SAME line

A | C | G | T
no newline here -> ...continued on same line


In [6]:
print("next page ->")
print("next page...", end=" ")
print('continued')


next page ->
next page... continued


**f-strings** embed variables (and formatting) directly in text — the modern, readable way.

In [7]:
# An f-string starts with f'...' and lets you drop variables inside { }.
asv = 'ASV_017'
gc = 0.531
length = 1465
# After the ':' you can add FORMAT instructions:
print(f'{asv}: {length} bp, GC = {gc:.1%}')   # '.1%' -> one-decimal percent
print(f'padded: |{asv:>12}|')                 # '>12' -> right-align in width 12

ASV_017: 1465 bp, GC = 53.1%
padded: |     ASV_017|


> **Instructor note.** f-strings are the single most useful day-to-day syntax. Encourage students to abandon `+`-based string building entirely.

## 2. String methods
Strings carry many built-in methods. The ones you will use on sequences:

In [8]:
# A 'method' is a function attached to a value, called with a dot: value.method()
s = '  atgGGCtttAAc  '   # deliberately messy: spaces + mixed case
print(s.strip())            # remove surrounding whitespace
print(s.strip().upper())    # methods can be CHAINED left-to-right
print('ATGCGC'.lower())     # force lower case
print('ATGCATGC'.count('ATG'))   # count occurrences of a motif
print('ATGCATGC'.find('CAT'))    # index of first match, or -1 if not found
print('ATGC'.startswith('ATG'))  # does it begin with this?

atgGGCtttAAc
ATGGGCTTTAAC
atgcgc
2
3
True


In [9]:
s = '  atgGGCtttAAc  '   # deliberately messy: spaces + mixed case
print(s.strip(), sep=' | ')

#didnt work

atgGGCtttAAc


`split` breaks a string into a list; `join` glues a list back together — essential for reading FASTA headers.

In [11]:
# FASTA headers are text we must chop up. split() is the tool.
header = '>ASV_017 Pseudomonas sp. | phylum=Pseudomonadota'
parts = header.split("|")          # split on whitespace -> a list of words
print(parts)
print('id  =', parts[0][1:])    # parts[0] is '>ASV_017'; [1:] drops the '>'
print('genus =', parts[1])      # the second word

# join() is the inverse: glue a list of strings with a separator.
print('-'.join(['ATG', 'GGC', 'TTT']))   # -> 'ATG-GGC-TTT'

['>ASV_017 Pseudomonas sp. ', ' phylum=Pseudomonadota']
id  = ASV_017 Pseudomonas sp. 
genus =  phylum=Pseudomonadota
ATG-GGC-TTT


## 3. Indexing
Positions start at **0**; negative indices count from the end.

In [18]:
# IMPORTANT: Python counts from 0, not 1. seq[0] is the FIRST base.
seq= 'ATGGGCTTTAACGGT'
print('first base :', seq[0])    # position 0
print('third base :', seq[2])    # position 2 = the 3rd base
print('last base  :', seq[-1])   # negative counts from the end: -1 = last
print('till last 5 bases :', seq[1:-5])
print('length     :', len(seq))

first base : A
third base : G
last base  : T
till last 5 bases : TGGGCTTTA
length     : 15


## 4. Slicing — `seq[start:stop:step]`
`stop` is **excluded**. A negative step reverses.

In [19]:
# Slicing takes a SUB-sequence: seq[start:stop]  (stop is NOT included).
print('first codon :', seq[0:3])   # bases 0,1,2 -> first codon
print('bases 3-8   :', seq[3:9])   # 3..8
print('from 6 on   :', seq[6:])    # omit stop -> to the end
print('every 3rd   :', seq[::3])   # the third number is the STEP
print('reversed    :', seq[::-1])  # step of -1 -> reverse the string

first codon : ATG
bases 3-8   : GGCTTT
from 6 on   : TTTAACGGT
every 3rd   : AGTAG
reversed    : TGGCAATTTCGGGTA


**Reverse complement** in one expression — reverse the strand and map each base to its partner. We use a translation table:

In [20]:
# str.maketrans builds a lookup: A<->T, G<->C (base pairing rules).
complement = str.maketrans('ACGT', 'TGCA')
# .translate() applies the complement; [::-1] reverses -> reverse complement.
rev_comp = seq.translate(complement)[::-1]
print('sequence        :', seq)
print('reverse comp.   :', rev_comp)

sequence        : ATGGGCTTTAACGGT
reverse comp.   : ACCGTTAAAGCCCAT


> **Instructor note.** Draw the two antiparallel strands on the board here; the `[::-1]` is the "read the other strand 5'->3'" step. This clicks once they see it.

## 5. Strings are immutable
You cannot change a character in place — you build a **new** string instead.

In [21]:
# Strings CANNOT be edited in place. This is a deliberate Python design.
dna = 'ATGCATG'
# dna[0] = 'C'          # <- uncommenting this raises TypeError

# Instead we BUILD A NEW string from pieces:
mutated = 'C' + dna[1:]   # new first base + everything from position 1 on
print('original:', dna)
print('mutated :', mutated)
print('via replace:', dna.replace('A', 'a'))  # replace() also returns a NEW string

original: ATGCATG
mutated : CTGCATG
via replace: aTGCaTG


## 6. Transcription & codons
Transcription replaces T with U; reading frame 1 splits the sequence into codons of three bases.

In [22]:
dna = 'ATGGGCTTTAACGGTTAA'

# Transcription DNA -> RNA is literally replacing every T with U:
rna = dna.replace('T', 'U')
print('DNA:', dna)
print('RNA:', rna)

# Split into codons (frame 1): take slices of 3 starting at 0, 3, 6, ...
# range(0, len-2, 3) gives those start positions and stops before a partial codon.
codons = [dna[i:i+3] for i in range(0, len(dna) - 2, 3)]
print('codons:', codons)

DNA: ATGGGCTTTAACGGTTAA
RNA: AUGGGCUUUAACGGUUAA
codons: ['ATG', 'GGC', 'TTT', 'AAC', 'GGT', 'TAA']


---
## Exercises
**E1.** For `seq = 'GATTACAGATTACA'`, print the first codon, the last codon, and the sequence reversed.

**E2.** Build the **reverse complement** of `seq` using the `translate` trick.

**E3.** Count how many times the motif `'GAT'` occurs, and report the index of its first occurrence.

**E4.** Using an f-string, print: `GATTACAGATTACA is 14 bp long (GC 29%)`.

In [26]:
seq = 'GATTACAGATTACA'
# your code here
print("reversed :" , seq[::-1])
print

reversed : ACATTAGACATTAG


<details>
<summary><b>Solution: E1-E4</b> (click to expand)</summary>

```python
seq = 'GATTACAGATTACA'
print(seq[:3], seq[-3:], seq[::-1])
comp = str.maketrans('ACGT', 'TGCA')
print(seq.translate(comp)[::-1])
print(seq.count('GAT'), seq.find('GAT'))
gc = (seq.count('G') + seq.count('C')) / len(seq)
print(f'{seq} is {len(seq)} bp long (GC {gc:.0%})')
```
</details>

### Recap
- **f-strings** `f'{value:.1%}'` format output cleanly.
- methods: `strip/upper/lower/count/find/split/join/replace`.
- slicing `[a:b:step]`; `[::-1]` reverses; `translate` + reverse = reverse complement.
- strings are **immutable**: operations return new strings.

Next: **03 · Data structures**.